In [1]:
import pymupdf4llm
loaded_document=pymupdf4llm.to_markdown("rm_frmt_nd_lwr.pdf")


Processing rm_frmt_nd_lwr.pdf...
[                                        ] (0/12[                                        ] (  1/12[                                        ] (  2/12[                                        ] (  3/129[=                                       ] (  4/129[=                                       ] (  5/129[=                                       ] (  6/12[==                                      ] (  7/12[==                                      ] (  8/12[==                                      ] (  9/129[===                                     ] ( 10/1[===                                     ] ( 11/129[===                                     ] ( 12/12[====                                    ] ( 13/12[====                                    ] ( 14/12[====                                    ] ( 15/12[====                                    ] ( 16/129[=====                                   ] ( 17/129[=====                                   ] ( 18/1[=====        

In [2]:
with open("man.md","w",encoding="utf-8") as f:
    f.write(loaded_document)


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# embeddings = HuggingFaceEmbeddings(model_name="nomic-ai/nomic-embed-text-v1.5",trust_remote_code=True)


d:\Chatbot\parent_retrival\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.storage._lc_store import create_kv_docstore
from langchain.storage import LocalFileStore
from langchain_chroma import Chroma

#vector_store
vector_store = Chroma(collection_name="HR_manual",embedding_function=embeddings,persist_directory="./RAG")
# create local_store
fs = LocalFileStore("./store_location")
store = create_kv_docstore(fs)


# define child splitter
child_splitter=RecursiveCharacterTextSplitter(
    chunk_size=250,  # must be less
    chunk_overlap=30,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
    # length_function=len
)

# Define parent splitter
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)


In [20]:
from langchain.retrievers import ParentDocumentRetriever
retriever = ParentDocumentRetriever(
    vectorstore=vector_store,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

In [21]:
vector_store.similarity_search("mobile policy")

[Document(id='c9adf9eb-d72c-4e73-83db-a168dfd1bee5', metadata={}, page_content='allotted mobile phone policy statement intent due exigency business sngpl requires certain member management remain accessible colleague customer office mobile phone therefore provided employee enable remain touch thereby promote smooth functioning company business paramount consideration'),
 Document(id='aed8af0a-2988-4928-8114-5d5cb020daa0', metadata={}, page_content='promote policy ensure easy accessibility shall circulated employee annual basis shall also placed company intranet awareness outside party synopsis policy along channel available raise concern shall placed company website stakeholder question content application policy may contact head internal'),
 Document(id='6c1b0708-d606-4e01-b8f3-15c0bcd8995a', metadata={}, page_content='mobile phone addition senior management company may allot mobile phone certain type position necessitate use management may also approve operational telephone lower gra

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os
load_dotenv()
api_key=os.getenv("GOOGLE_API_KEY")
llm=ChatGoogleGenerativeAI(model="gemini-1.5-flash",api_key=api_key)

In [11]:
# loaded_document
from langchain_community.document_loaders import UnstructuredMarkdownLoader
loader = UnstructuredMarkdownLoader("man.md")
data = loader.load()


In [12]:
%pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [17]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from langchain_community.document_loaders import UnstructuredMarkdownLoader

# ✅ Download necessary NLTK resources
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")

def preprocess_text(text):
    """Cleans and preprocesses text before further use."""
    tokens = word_tokenize(text.lower())  # Lowercasing & tokenization
    tokens = [word for word in tokens if word.isalnum()]  # Remove punctuation
    tokens = [word for word in tokens if word not in stopwords.words("english")]  # Remove stopwords
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]  # Lemmatization
    return " ".join(tokens)

# ✅ Load raw document
loader = UnstructuredMarkdownLoader("man.md")
data = loader.load()

# ✅ Extract and preprocess text
raw_texts = [doc.page_content for doc in data]
preprocessed_texts = [preprocess_text(text) for text in raw_texts]

# ✅ Output preprocessed text (for debugging)
for i, text in enumerate(preprocessed_texts):
    print(f"Processed Document {i+1}:")
    print(text[:500])  # Print first 500 chars of each processed document
    print("-" * 80)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\husnain.mahmood\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\husnain.mahmood\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\husnain.mahmood\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Processed Document 1:
executive summary executive service rule first introduced may 1981 amended per bod approval time time currently applicable direction hr r nc bod service hr consultant commissioned review assess suitability existing rule per current market condition propose amendment policy practice document prepare new manual financial admin power position level also made integral part hr manual per advice chairman hr r nc board director approved hr manual 344th meeting held addition amendment different clause t
--------------------------------------------------------------------------------


In [11]:
retriever.add_documents(data)

In [ ]:
query="what is the maximum age required by grade iv?"
result=retriever.invoke(query.lower())
# result=vector_store.similarity_search(query.lower())
llm.invoke(f"you are assistant that tell that the context given to you provide accurate answer. just reply with yes or no. this in context:{result} and this is query {query}").content


'Yes'